In [0]:
from pyspark import pipelines as dp
import re

from pyspark.sql.functions import col, concat_ws, when

In [0]:
# Silver transformations on the bronze customers table
@dp.materialized_view(name="customers_To_silver", comment="Cleaned customers: UserName, no Phone 2, Discount, snake_case columns")
def customers_silver():
    #Broze Table
    df = spark.read.table("dawood.dawood.customers_2000000")

    df_silver = (
        df
        .withColumn("UserName", concat_ws(" ", col("First Name"), col("Last Name")))
        .drop("First Name", "Last Name")
        .drop("Phone 2")
        .withColumn(
            "Discount",
            when(col("Country") == "UUnited Kingdom", "30%").otherwise("10%"),
        )
        .filter(col("Subscription Date").isNotNull())
        .orderBy(col("Subscription Date").desc())
    )

    # Rename all columns to snake_case (e.g. "Customer Id" -> "customer_id")
    snake_case_columns = [re.sub(r"\s+", "_", c.strip()).lower() for c in df_silver.columns]
    df_silver = df_silver.toDF(*snake_case_columns)

    return df_silver

In [0]:
# customers_silver is published automatically to the pipeline's catalog/schema
# on each update — no manual write/saveAsTable needed.